# 📋 NLP en Reportes Médicos — Reconocimiento de Entidades
## Taller Pre-Congreso CNIB 2025

*Inspirado en la clase de NLP de la Dra. Helena Gómez Adorno y Diego Hernández Bustamante — Escuela Híbrida de Verano "Medical Informatics with AI" UNAM-TUBS*

---

> **¿Qué es el NLP (Procesamiento de Lenguaje Natural)?** Es el área de la IA que permite a las computadoras entender, interpretar y generar texto humano. En medicina, el NLP es fundamental porque la mayoría de la información clínica está en texto libre: notas del médico, reportes de radiología, resúmenes de alta, etc.

> **Dato impactante 📊:** Se estima que entre el 70-80% de los datos médicos existen exclusivamente en formato de texto no estructurado (notas clínicas). Sin NLP, esa información es inaccesible para análisis a gran escala.

---


In [ ]:
# Instalamos spaCy y descargamos el modelo de español
!pip -q install spacy
!python -m spacy download es_core_news_md

In [ ]:
# Expedientes Médicos Electrónicos (EMR) de ejemplo — reportes clínicos ficticios en español
emr = [
"""Paciente mujer de 45 años. Niega fiebre. Refiere dolor torácico desde hace 2 días y disnea leve.
Se indica paracetamol 500 mg cada 8 h. Examen: crepitantes en base pulmonar derecha.""",
"""Varón de 62 años con DM2 en control. Sin tos ni cefalea. Se prescribe metformina 850 mg 2 veces al día.
Dolor abdominal intermitente; descarta vómito. Amoxicilina 500 mg c/12h por 7 días.""",
"""Paciente masculino 30 años. Faringitis; ibuprofeno 400mg cada 8h. Niega disnea. Ausencia de fiebre.
Dolor en garganta y adenopatías cervicales."""
]

---

## 🏷️ NER sin Machine Learning — Basado en Reglas

**NER (Named Entity Recognition):** Tarea de identificar y clasificar entidades nombradas en texto. En medicina: medicamentos, síntomas, dosis, anatomía, etc.

**Enfoque basado en reglas:** Definimos listas de términos y patrones lingüísticos explícitos. Ventaja: totalmente controlable y explicable. Desventaja: no generaliza a términos no listados.

**spaCy EntityRuler:** Permite agregar patrones personalizados al pipeline de NLP de spaCy sin necesitar un modelo entrenado.

> **Contexto clínico:** Los sistemas de NER médico son la base de la farmacovigilancia automatizada — detectar menciones de medicamentos y efectos adversos en millones de registros clínicos para identificar señales de seguridad que ningún humano podría revisar manualmente.


In [ ]:
import re, json
import spacy
from spacy.pipeline import EntityRuler

# Cargamos el modelo de español (mediano)
# exclude=["ner"]: excluimos el reconocedor de entidades genérico de spaCy
# porque vamos a agregar el nuestro personalizado
nlp = spacy.load("es_core_news_md", exclude=["ner"])

In [ ]:
# Definimos nuestros "diccionarios médicos" por categoría

MEDICAMENTOS = ["paracetamol", "metformina", "amoxicilina", "ibuprofeno",
                "naproxeno", "omeprazol"]

SINTOMAS     = ["fiebre", "tos", "cefalea", "disnea", "vómito", "dolor",
                "faringitis", "adenopatías", "náusea"]

ANATOMIA     = ["pulmonar", "pulmón", "torácico", "garganta", "abdomen",
                "cervicales", "corazón"]

In [ ]:
# Función auxiliar: convierte una lista de términos en patrones de spaCy
# Cada patrón es: {"label": "CATEGORIA", "pattern": [{"LOWER": "termino"}]}
# LOWER: hace la búsqueda insensible a mayúsculas

def dict_patterns(terms, label):
    return [{"label": label, "pattern": [{"LOWER": t.lower()}]} for t in terms]

---

## 📋 Construcción de la Lista de Patrones


In [ ]:
# Lista de patrones para el EntityRuler
patterns = []

In [ ]:
# Patrones de medicamentos
patterns += dict_patterns(MEDICAMENTOS, "MEDICAMENTO")
print("Patrones de medicamentos:")
print(patterns[:3], "...")

In [ ]:
# Patrones de dosis: número + unidad (ej: "500 mg", "10 ml")
# REGEX en spaCy: \d+ = uno o más dígitos, (\.\d+)? = decimales opcionales
patterns += [{
    "label": "DOSIS",
    "pattern": [
        {"TEXT": {"REGEX": r"^\d+(\.\d+)?$"}},  # número
        {"LOWER": {"IN": ["mg", "ml", "mcg"]}}       # unidad
    ]
}]
print("Patrón de dosis con espacio (ej: '500 mg') agregado.")

In [ ]:
# Patrones de dosis sin espacio (ej: "400mg", "5ml")
patterns += [{
    "label": "DOSIS",
    "pattern": [{"TEXT": {"REGEX": r"^\d+(\.\d+)?(mg|ml|mcg)$"}}]
}]
print("Patrón de dosis sin espacio (ej: '400mg') agregado.")

In [ ]:
# Patrones de frecuencia de administración (ej: "cada 8 horas", "c/12h", "2 veces al día")
patterns += [{
    "label": "FRECUENCIA",
    "pattern": [
        {"LOWER": {"IN": ["cada", "c/", "veces"]}},
        {"IS_ALPHA": True, "OP": "*"}   # OP="*" = cero o más tokens
    ]
}]
print("Patrón de frecuencia agregado.")

In [ ]:
# Patrones de síntomas y anatomía
patterns += dict_patterns(SINTOMAS, "SINTOMA")
patterns += dict_patterns(ANATOMIA, "ANATOMIA")

# Hallazgo físico específico (crepitantes = sonido pulmonar anormal, signo de neumonía)
patterns += [{"label": "HALLAZGO", "pattern": "crepitantes"}]

print(f"Total de patrones definidos: {len(patterns)}")

---

## 🔧 Agregar EntityRuler al Pipeline de spaCy


In [ ]:
# Agregamos el EntityRuler al pipeline de spaCy
# overwrite_ents=True: nuestras entidades tienen prioridad sobre las del modelo base
ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})
ruler.add_patterns(patterns)

print(f"Pipeline de NLP: {nlp.pipe_names}")

---

## 🔍 Procesamiento de los Reportes Médicos


In [ ]:
# Procesamos cada reporte y mostramos las entidades detectadas
# start_char, end_char: posición del texto original donde aparece la entidad

for i, txt in enumerate(emr, 1):
    doc = nlp(txt)
    print(f"\n--- REPORTE {i} ---")
    print(txt)
    print("\nEntidades detectadas:")
    for ent in doc.ents:
        print(f"  [{ent.label_}] '{ent.text}' (posición {ent.start_char}-{ent.end_char})")

---

## 🔗 Extracción de Relaciones y Contexto

Ahora vamos más allá del NER básico: detectamos **negaciones** y **relaciones** entre entidades (medicamento↔dosis).

### Detección de Negaciones

En texto médico, "niega fiebre" y "presenta fiebre" son opuestos con consecuencias clínicas totalmente diferentes. Detectar negaciones es crucial.

**Pistas de negación (Negation Cues):** palabras que niegan lo que viene después.


In [ ]:
# Lista de palabras que indican negación en el lenguaje médico en español
NEG_CUES = ["no", "sin", "niega", "descarta", "ausencia de", "niega tener", "no refiere"]

In [ ]:
def detect_negation(doc, NEG_CUES, target_labels=("SINTOMA", "HALLAZGO", "PROBLEMA")):
    """
    Detecta si un síntoma o hallazgo está negado en el texto.
    Algoritmo: para cada entidad del tipo buscado, revisa si hay una pista de negación
    en las 5 palabras anteriores dentro de la misma oración.
    """
    results = []
    for ent in doc.ents:
        if ent.label_ in target_labels:
            sent   = ent.sent
            # Ventana de texto antes de la entidad (en la misma oración)
            window = doc[max(sent.start, ent.start - 5): ent.start].text.lower()
            negated = any(cue in window for cue in NEG_CUES)
            results.append({
                "text":    ent.text,
                "label":   ent.label_,
                "start":   ent.start_char,
                "end":     ent.end_char,
                "negated": negated
            })
    return results

In [ ]:
def link_med_dose(doc, max_window=8):
    """
    Relaciona medicamentos con su dosis y frecuencia.
    Heurística: busca la dosis más cercana al medicamento dentro de la misma oración.
    """
    pairs = []
    meds  = [e for e in doc.ents if e.label_ == "MEDICAMENTO"]
    doses = [e for e in doc.ents if e.label_ == "DOSIS"]
    freqs = [e for e in doc.ents if e.label_ == "FRECUENCIA"]

    for med in meds:
        # Candidatos: dosis en la misma oración dentro del rango de ventana
        cand_dose = [(dose, abs(dose.start - med.start))
                     for dose in doses
                     if dose.sent == med.sent and abs(dose.start - med.start) <= max_window]
        if cand_dose:
            dose = sorted(cand_dose, key=lambda x: x[1])[0][0]
            cand_freq = [(f, abs(f.start - med.start))
                         for f in freqs
                         if f.sent == med.sent and abs(f.start - med.start) <= max_window]
            freq = sorted(cand_freq, key=lambda x: x[1])[0][0].text if cand_freq else ""
            pairs.append((med.text, dose.text, freq))
    return pairs

In [ ]:
# Procesamos y mostramos negaciones y relaciones para cada reporte
for i, txt in enumerate(emr, 1):
    doc = nlp(txt)
    neg = detect_negation(doc, NEG_CUES)
    rel = link_med_dose(doc)

    print(f"\n=== REPORTE {i} ===")
    print("\nSíntomas y su estado (afirmado/negado):")
    for x in neg:
        flag = "❌ NEGADO" if x["negated"] else "✅ AFIRMADO"
        print(f"  {x['text']} ({x['label']}) → {flag}")

    print("\nPares Medicamento — Dosis — Frecuencia:")
    for med, dose, freq in rel:
        print(f"  💊 {med} ↔ {dose}" + (f" ↔ {freq}" if freq else ""))

---

## 🤖 NER con Transformers (BERT en Español)

**Modelos de Transformers:** Los transformers (como BERT) aprenden representaciones contextuales del lenguaje. A diferencia de nuestro enfoque basado en reglas, un modelo entrenado puede reconocer entidades que *no* están en el diccionario, basándose en el contexto.

**`lcampillos/roberta-es-clinical-trials-ner`:** Modelo entrenado específicamente en textos de ensayos clínicos en español. Reconoce entidades biomédicas con mucha mayor cobertura que nuestro sistema basado en reglas.

> **Historia del NLP médico 📚:** BERT fue publicado por Google en 2018. En 2019 apareció BioBERT (entrenado en PubMed). En 2020, ClinicalBERT (entrenado en notas clínicas). Hoy existen docenas de variantes para idiomas y especialidades específicas.


In [ ]:
!pip -q install transformers torch accelerate

In [ ]:
from transformers import pipeline

# Cargamos un modelo NER pre-entrenado para textos clínicos en español
# Este modelo usa RoBERTa (variante robusta de BERT) fine-tuned en ensayos clínicos
pipe = pipeline(
    "token-classification",
    model="lcampillos/roberta-es-clinical-trials-ner"
)
print("Modelo cargado.")

In [ ]:
# Aplicamos el modelo de Transformers a cada reporte
# El modelo predice una etiqueta BIO (Beginning-Inside-Outside) para cada token

for i, txt in enumerate(emr, 1):
    print(f"\n=== REPORTE {i} ===")
    results = pipe(txt)
    for ent in results:
        # Limpiamos artefactos de tokenización del modelo
        text = ent["word"].replace('Ġ','').replace('Ã¡','á').replace('Ã³','ó').replace('ÃŃ','ó')
        # Extraemos el tipo de entidad (quitando el prefijo B- o I-)
        entity_type = ent['entity'].split('-')[1] if '-' in ent['entity'] else ent['entity']
        print(f"  {text:<15} → {entity_type}  (score: {ent['score']:.3f})")

---

## 🌐 Usando LLMs — Google Gemini para NLP Médico

**LLMs (Large Language Models):** Modelos masivos como GPT, Gemini o Claude, entrenados en enormes cantidades de texto. Tienen capacidades de NLP generalistas: traducción, resumen, extracción de información, respuesta a preguntas.

**Ventaja sobre modelos especializados:** No requieren fine-tuning — basta con un buen *prompt*. Entienden instrucciones en lenguaje natural.

**Desventaja:** Costo computacional mayor, posibles alucinaciones, privacidad de datos del paciente.

> **Contexto de privacidad ⚠️:** En medicina, enviar datos de pacientes reales a APIs externas como Gemini o GPT podría violar la HIPAA (EE.UU.) o la LGPD (México). En producción, se usan modelos on-premise o se anonimiza el texto antes de enviarlo.

**Esquema de clasificación que usamos (basado en MeSH/BioNER):**
- `ANAT`: partes del cuerpo y anatomía
- `CHEM`: medicamentos y sustancias químicas
- `DISO`: condiciones patológicas y síntomas
- `PROC`: procedimientos diagnósticos y terapéuticos


In [ ]:
from google.colab import userdata
from google import genai
import os

# ⚠️ En un ambiente real, usa variables de entorno o un vault de secretos
# NUNCA pongas API keys directamente en el código de producción
os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"

# client = genai.Client(api_key=userdata.get('gemini_key'))  # Opción para Colab con secrets
client = genai.Client(api_key="YOUR_API_KEY")

In [ ]:
# Prompt en inglés para Gemini: instrucciones claras de la tarea NER
context = """Given the following texts, return medical entities you could find,
classify them in 4 semantic groups:
  ANAT: body parts and anatomy (e.g. garganta, 'throat')
  CHEM: chemical entities and pharmacological substances (e.g. aspirina, 'aspirin')
  DISO: pathologic conditions (e.g. dolor, 'pain')
  PROC: diagnostic and therapeutic procedures (e.g. cirugía, 'surgery')"""

# Configuraciones de seguridad permisivas para textos médicos técnicos
# (los filtros de contenido a veces bloquean términos médicos como 'dolor', 'muerte')
safety_settings = [
    {"category": "HARM_CATEGORY_DANGEROUS",        "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT",        "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH",       "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
]

In [ ]:
# Enviamos cada reporte a Gemini y mostramos las entidades detectadas
for txt in emr:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=context + '\n' + txt
    )
    print('\n' + '='*60)
    print(response.text)

---

## 📊 Comparativa de los Tres Enfoques

| Enfoque | Ventajas | Desventajas |
|---|---|---|
| **Reglas (spaCy)** | Controlable, explicable, sin entrenamiento | Solo reconoce lo que está en el diccionario |
| **Transformers (BERT/RoBERTa)** | Generaliza a términos nuevos, contexto semántico | Requiere GPU, modelo específico por idioma/dominio |
| **LLM (Gemini/GPT)** | Sin entrenamiento, instrucciones en lenguaje natural | Costo, privacidad, posibles alucinaciones |

> **Recomendación práctica 💡:** Para producción clínica con datos reales de pacientes, combina reglas (para entidades críticas y seguras) + un modelo Transformer local (para cobertura amplia) + revisión humana de expertos.
